# TT Means Test time trained
This notebook contains only the Qwen 2.5 0.5B model fine-tuning and inference. It is fine-tuned on the test data during test


**Note: This submission uses only the 0.5B Qwen model for standalone classification.**

Can be used to experiment how different models perform on Test-time training

# Original Notebook(1)
https://www.kaggle.com/code/kishanvavdara/test-on-testdataset-qwenemdding-llama-lr

Only the .5B model part was separated ,nothing else changed

In [ ]:
!uv pip install --system --no-index --find-links='/kaggle/input/jigsaw-packages2/whls/' 'trl==0.21.0' 'optimum==1.27.0' 'auto-gptq==0.7.1' 'bitsandbytes==0.46.1' 'deepspeed==0.17.4' 'logits-processor-zoo==0.2.1' 'vllm==0.10.0'
!uv pip install --system --no-index --find-links='/kaggle/input/jigsaw-packages2/whls/' 'triton==3.2.0'
!uv pip install --system --no-index --find-links='/kaggle/input/jigsaw-packages2/whls/' 'clean-text'
!uv pip install --system --no-index -U --no-deps --find-links='/kaggle/input/jigsaw-packages2/whls/' 'peft' 'accelerate' 'datasets'

# 1. Test time train Qwen 2.5 0.5b

In [ ]:
#in v1, takes full test set for traninig then infers on 167k only cureted reddits for pretranining(no 31k),sends them to deberta model for 2 epochs on unsup, then 8 epochs supervised

In [ ]:
import pandas as pd
select= pd.read_csv('/kaggle/input/199kv2-jigsaw/test.csv')
select.iloc[31750:].to_csv('test.csv',index=False)
# select.iloc[-10:].to_csv('test.csv',index=False)

In [ ]:
%%writefile constants.py
BASE_MODEL_PATH = "/kaggle/input/5b-trained/merged"
LORA_PATH = "output/"
DATA_PATH = "/kaggle/input/jigsaw-agile-community-rules/"

POSITIVE_ANSWER = "Yes"
NEGATIVE_ANSWER = "No"
COMPLETE_PHRASE = "Answer:"
BASE_PROMPT = '''You are given a comment from reddit and a rule. Your task is to classify whether the comment violates the rule. Only respond Yes/No.'''

In [ ]:
%%writefile utils.py
import pandas as pd
from datasets import Dataset
from constants import POSITIVE_ANSWER, NEGATIVE_ANSWER, COMPLETE_PHRASE, BASE_PROMPT
import random, numpy as np
random.seed(42)
np.random.seed(42)


def build_prompt(row):
    return f"""
{BASE_PROMPT}

Subreddit: r/{row["subreddit"]}
Rule: {row["rule"]}
Examples:
1) {row["positive_example"]}
{COMPLETE_PHRASE} Yes

2) {row["negative_example"]}
{COMPLETE_PHRASE} No

---
Comment: {row["body"]}
{COMPLETE_PHRASE}"""


def get_dataframe_to_train(data_path):
    train_dataset = pd.read_csv(f"{data_path}/train.csv")
    test_dataset = pd.read_csv(f"{data_path}/test.csv").reset_index(drop=True)

    flatten = []

    # ---------- 处理训练集 ----------
    train_df = train_dataset[["body", "rule", "subreddit", "rule_violation",
                              "positive_example_1","positive_example_2",
                              "negative_example_1","negative_example_2"]].copy()

    # 随机选 positive_example 和 negative_example
    train_df["positive_example"] = np.where(
        np.random.rand(len(train_df)) < 0.5,
        train_df["positive_example_1"],
        train_df["positive_example_2"]
    )
    train_df["negative_example"] = np.where(
        np.random.rand(len(train_df)) < 0.5,
        train_df["negative_example_1"],
        train_df["negative_example_2"]
    )

    # 删除原来的候选列
    train_df.drop(columns=["positive_example_1","positive_example_2",
                           "negative_example_1","negative_example_2"], inplace=True)

    flatten.append(train_df)

    # ---------- 处理测试集 ----------
    for violation_type in ["positive", "negative"]:
        for i in range(1, 3):
            sub_dataset = test_dataset[["rule","subreddit",
                                        "positive_example_1","positive_example_2",
                                        "negative_example_1","negative_example_2"]].copy()

            if violation_type == "positive":
                # body 用当前 positive_example
                body_col = f"positive_example_{i}"
                other_positive_col = f"positive_example_{3-i}"  # 另一个 positive
                sub_dataset["body"] = sub_dataset[body_col]
                sub_dataset["positive_example"] = sub_dataset[other_positive_col]
                # negative_example 随机选
                sub_dataset["negative_example"] = np.where(
                    np.random.rand(len(sub_dataset)) < 0.5,
                    sub_dataset["negative_example_1"],
                    sub_dataset["negative_example_2"]
                )
                sub_dataset["rule_violation"] = 1

            else:  # violation_type == "negative"
                body_col = f"negative_example_{i}"
                other_negative_col = f"negative_example_{3-i}"
                sub_dataset["body"] = sub_dataset[body_col]
                sub_dataset["negative_example"] = sub_dataset[other_negative_col]
                sub_dataset["positive_example"] = np.where(
                    np.random.rand(len(sub_dataset)) < 0.5,
                    sub_dataset["positive_example_1"],
                    sub_dataset["positive_example_2"]
                )
                sub_dataset["rule_violation"] = 0

            # 删除原来的候选列
            sub_dataset.drop(columns=["positive_example_1","positive_example_2",
                                      "negative_example_1","negative_example_2"], inplace=True)

            flatten.append(sub_dataset)

    # 合并所有 DataFrame
    dataframe = pd.concat(flatten, axis=0)
    dataframe = dataframe.drop_duplicates(ignore_index=True)

    return dataframe



def build_dataset(dataframe):
    dataframe["prompt"] = dataframe.apply(build_prompt, axis=1)

    columns = ["prompt"]
    if "rule_violation" in dataframe:
        dataframe["completion"] = dataframe["rule_violation"].map(
            {
                1: POSITIVE_ANSWER,
                0: NEGATIVE_ANSWER,
            }
        )
        columns.append("completion")

    dataframe = dataframe[columns]
    dataset = Dataset.from_pandas(dataframe)
    dataset.to_pandas().to_csv("/kaggle/working/dataset.csv", index=False)
    return dataset

In [ ]:
%%writefile train.py
import pandas as pd

from trl import SFTTrainer, SFTConfig
from peft import LoraConfig,PeftModel,get_peft_model
from tqdm.auto import tqdm
from transformers.utils import is_torch_bf16_gpu_available
from utils import build_dataset, get_dataframe_to_train
from constants import DATA_PATH, BASE_MODEL_PATH, LORA_PATH

def main():
    dataframe = get_dataframe_to_train(DATA_PATH)
    train_dataset = build_dataset(dataframe)
    
    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.1,
        bias="none",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        task_type="CAUSAL_LM",
    )
    
    training_args = SFTConfig(
        num_train_epochs=1,
        
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        
        optim="paged_adamw_8bit",
        learning_rate=2e-4, #keep high, lora usually likes high. 
        weight_decay=0.01,
        max_grad_norm=1.0,
        
        lr_scheduler_type="cosine",
        warmup_ratio=0.03,
        
        fp16=True,
        dataloader_pin_memory=True,
        
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
    
        save_strategy="no",
        report_to="none",
    
        completion_only_loss=True,
        packing=False,
        remove_unused_columns=False,
    )
    
    trainer = SFTTrainer(
        BASE_MODEL_PATH,
        args=training_args,
        train_dataset=train_dataset,
        peft_config=lora_config,
    )
    
    trainer.train()
    trainer.save_model(LORA_PATH)
    # MERGE TT LoRA into distilled base
    print("Merging TT LoRA into distilled base model...")
    merged_model = trainer.model.merge_and_unload()
    merged_model.save_pretrained("final_merged_model/")
    
    # Save tokenizer
    from transformers import AutoTokenizer
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_PATH)
    tokenizer.save_pretrained("final_merged_model/")
    
    print("✓ Final merged model saved to final_merged_model/")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile inference.py
import os
os.environ["VLLM_USE_V1"] = "0"

import vllm
import pandas as pd
from logits_processor_zoo.vllm import MultipleChoiceLogitsProcessor
from utils import build_dataset
from constants import DATA_PATH, POSITIVE_ANSWER, NEGATIVE_ANSWER
import random
import multiprocessing as mp
from scipy.special import softmax

FINAL_MODEL_PATH = "final_merged_model/"  # ← Use merged model, not LoRA


def run_inference_on_device(df_slice):
  llm = vllm.LLM(
      FINAL_MODEL_PATH,  # ← Merged model path
      # quantization="gptq",  # ← REMOVE (FP16 model)
      dtype="float16",  # ← ADD for FP16
      tensor_parallel_size=1,
      gpu_memory_utilization=0.85,
      trust_remote_code=True,
      max_model_len=3000,
      disable_log_stats=True,
      enable_prefix_caching=True,
      # enable_lora=True,  # ← REMOVE (no LoRA needed)
      # max_lora_rank=64,  # ← REMOVE
      max_num_seqs=32,  # Limit concurrent sequences
      max_num_batched_tokens=8192,  # Limit batch token count

  )

  tokenizer = llm.get_tokenizer()
  mclp = MultipleChoiceLogitsProcessor(tokenizer, choices=[POSITIVE_ANSWER, NEGATIVE_ANSWER])

  test_dataset = build_dataset(df_slice)
  texts = test_dataset["prompt"]

  outputs = llm.generate(
      texts,
      vllm.SamplingParams(
          skip_special_tokens=True,
          max_tokens=1,
          logits_processors=[mclp],
          logprobs=2,
      ),
      use_tqdm=True,
      # lora_request=LoRARequest("default", 1, LORA_PATH)  # ← REMOVE
  )

  log_probs = [
      {lp.decoded_token: lp.logprob for lp in out.outputs[0].logprobs[0].values()}
      for out in outputs
  ]
  predictions = pd.DataFrame(log_probs)[[POSITIVE_ANSWER, NEGATIVE_ANSWER]]
  predictions[[POSITIVE_ANSWER, NEGATIVE_ANSWER]] = predictions[[POSITIVE_ANSWER, NEGATIVE_ANSWER]].apply(lambda x: softmax(x.values), axis=1, result_type="expand")
  predictions["row_id"] = df_slice["row_id"].values
  return predictions


def worker(device_id, df_slice, return_dict):
    # 限制该进程只看到一张 GPU
    os.environ["CUDA_VISIBLE_DEVICES"] = str(device_id)
    print(f"[Worker {device_id}] Running on GPU {device_id}, data size={len(df_slice)}")

    preds = run_inference_on_device(df_slice)
    return_dict[device_id] = preds


def main():
    test_dataframe = pd.read_csv('test.csv')

    # 随机选择例子
    test_dataframe["positive_example"] = test_dataframe.apply(
        lambda row: random.choice([row["positive_example_1"], row["positive_example_2"]]),
        axis=1
    )
    test_dataframe["negative_example"] = test_dataframe.apply(
        lambda row: random.choice([row["negative_example_1"], row["negative_example_2"]]),
        axis=1
    )
    test_dataframe = test_dataframe.drop(
        columns=["positive_example_1", "positive_example_2", "negative_example_1", "negative_example_2"],
        errors="ignore"
    )

    # 切分数据
    mid = len(test_dataframe) // 2
    df0 = test_dataframe.iloc[:mid].reset_index(drop=True)
    df1 = test_dataframe.iloc[mid:].reset_index(drop=True)

    manager = mp.Manager()
    return_dict = manager.dict()

    # 两个进程并行
    p0 = mp.Process(target=worker, args=(0, df0, return_dict))
    p1 = mp.Process(target=worker, args=(1, df1, return_dict))
    p0.start()
    p1.start()
    p0.join()
    p1.join()

    # 合并结果
    predictions = pd.concat([return_dict[0], return_dict[1]], ignore_index=True)

    # 构建 submission
    # submission = predictions[["row_id", POSITIVE_ANSWER]].rename(columns={POSITIVE_ANSWER: "rule_violation"})
    # rq = submission['rule_violation']#.rank(method='average') / (len(submission) + 1)
    # submission['rule_violation'] = rq

    test_dataframe= test_dataframe.merge(predictions[["row_id", POSITIVE_ANSWER]],on='row_id',how='left')
    test_dataframe['rule_violation']= test_dataframe[POSITIVE_ANSWER]
    test_dataframe.to_csv('llmtest.csv',index=False)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile accelerate_config.yaml
compute_environment: LOCAL_MACHINE
debug: false
deepspeed_config:
  gradient_accumulation_steps: 4
  gradient_clipping: 1.0
  train_batch_size: 64
  train_micro_batch_size_per_gpu: 4
  
  zero_stage: 2
  offload_optimizer_device: none
  offload_param_device: none
  zero3_init_flag: false
  
  stage3_gather_16bit_weights_on_model_save: false
  stage3_max_live_parameters: 1e8
  stage3_max_reuse_distance: 1e8
  stage3_prefetch_bucket_size: 5e7
  stage3_param_persistence_threshold: 1e5
  
  zero_allow_untested_optimizer: true
  zero_force_ds_cpu_optimizer: false
  
  fp16:
    enabled: true
    loss_scale: 0
    initial_scale_power: 16
    loss_scale_window: 1000
    hysteresis: 2
    min_loss_scale: 1
  
distributed_type: DEEPSPEED
downcast_bf16: 'no'
dynamo_config:
  dynamo_backend: INDUCTOR
  dynamo_use_fullgraph: false
  dynamo_use_dynamic: false
enable_cpu_affinity: false
machine_rank: 0
main_training_function: main
mixed_precision: fp16
num_machines: 1
num_processes: 2
rdzv_backend: static
same_network: true
tpu_env: []
tpu_use_cluster: false
tpu_use_sudo: false
use_cpu: false

In [ ]:
!accelerate launch --config_file accelerate_config.yaml train.py

In [ ]:
!python inference.py

In [ ]:
import os
import torch
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from transformers import AutoTokenizer, AutoModel, get_cosine_schedule_with_warmup
from tqdm import tqdm
import multiprocessing as mp
from transformers import get_linear_schedule_with_warmup

# %env KAGGLE_IS_COMPETITION_RERUN ='true'

In [ ]:
MAX_LEN = 256//2
BATCH_SIZE = 16*2
EPOCHS =  8
U_EPOCHS = 2
MODEL_PATH = "/kaggle/input/deberta-v3-base/transformers/default/1/deberta-v3-base"
SEEDS = [42, 123]

In [ ]:
train_path = "/kaggle/input/jigsaw-agile-community-rules/train.csv"
test_path = "/kaggle/input/jigsaw-agile-community-rules/test.csv"
sample_sub_path = "/kaggle/input/jigsaw-agile-community-rules/sample_submission.csv"

In [ ]:
df = pd.read_csv(train_path)
df['rule']= df['rule'].str.lower().str.strip()
df["text"] = df["rule"] + " [SEP] " + df["body"]
df["label"] = df["rule_violation"].astype(float)

In [ ]:
def add_data(dataframe):
    ret=[[],[]]
    for i in ['positive_example_1','positive_example_2','negative_example_1','negative_example_2']:
        tmp= (dataframe['rule']+' [SEP] '+ dataframe[i]).tolist()
        ret[0]+= tmp
        ret[1]+= [1]*len(tmp) if 'positive' in i else [0]*len(tmp)
    return ret

In [ ]:
test_df = pd.read_csv(test_path)
test_df['rule']= test_df.rule.str.lower().str.strip()

augmented_train = add_data(df)
augmented_test = add_data(test_df)

augmented_texts =  df.text.tolist()+augmented_train[0] + augmented_test[0]
augmented_labels =  df.label.tolist()+augmented_train[1] + augmented_test[1]

augmented_df = pd.DataFrame({
    'text': augmented_texts,
    'label': augmented_labels
})
print(f'Before:{augmented_df.shape}')
augmented_df = augmented_df.groupby(augmented_df['text'].str.lower(), as_index=False).agg({
    'text': 'first',
    'label': 'mean'
})
print('After:',augmented_df.shape)
augmented_df['rule']= augmented_df.text.apply(lambda x: x.split(' [SEP] ')[0])
augmented_df['body']= augmented_df.text.apply(lambda x: x.split(' [SEP] ')[1])


augmented_df.head()

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, use_fast = False)

In [ ]:
unlabelled= pd.read_csv('llmtest.csv')
unlabelled['rule']= unlabelled['rule'].str.lower().str.strip()
unlabelled['text']= unlabelled['rule']+ ' [SEP] '+ unlabelled['body']

rule_map= {i:j for j,i in enumerate(unlabelled.rule.unique())}
augmented_df['rule_id']= augmented_df.rule.map(rule_map)
unlabelled['rule_id']= unlabelled.rule.map(rule_map)

In [ ]:
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    for seed in SEEDS:
        train_data, val_data = train_test_split(
            augmented_df, 
            test_size=0.2, 
            stratify=augmented_df["rule"], 
            random_state=seed
        )
        unlabelled_seed= unlabelled.copy()
        unlabelled_seed['label']= unlabelled_seed.rule_violation.round(0)
        # # change to keep the actual test set out of llm predictions.
        # unlabelled_seed= unlabelled_seed.query('text not in @augmented_df.text.values')

        temp_test_data= test_df["rule"].str.lower().str.strip() + " [SEP] " + test_df["body"]
        unlabelled_seed= unlabelled_seed.query('text not in @temp_test_data')

        # confident_positives = unlabelled_seed[unlabelled_seed['rule_violation'] >= 0.85]

        # # Take confident negatives (pred <= 0.1)
        # confident_negatives = unlabelled_seed[unlabelled_seed['rule_violation'] <= 0.15]
        
        # # Sample same number of negatives + 10%
        # n_positives = len(confident_positives)
        # n_negatives_to_sample = int(n_positives * 1.1)
        
        # sampled_negatives = confident_negatives.sample(
        #   n=min(n_negatives_to_sample, len(confident_negatives)),
        #   random_state=seed
        # )
        
        # # Combine
        # unlabelled_taken = pd.concat([confident_positives, sampled_negatives], ignore_index=True)
        unlabelled_taken= unlabelled_seed
        
        unlabelled_taken= unlabelled_taken.sample(frac=.8, random_state=seed)# take 80% of all the points for data diversity
        # print(f'Seed {seed} - Pseudo: {len(confident_positives)} pos (>=0.85), {len(sampled_negatives)} neg (<=0.15), total={len(unlabelled_taken)}')

        train_data.to_csv(f'fixed_train_split_seed_{seed}.csv', index=False)
        val_data.to_csv(f'fixed_val_split_seed_{seed}.csv', index=False)
        unlabelled_taken.to_csv(f'fixed_pseudo_seed_{seed}.csv', index=False)
        print(f'Seed {seed} splits saved: train={len(train_data)}, val={len(val_data)}, pseudo={len(unlabelled_taken)}')

In [ ]:
class JigsawDataset(Dataset):
    def __init__(self, texts, labels,rule_ids, tokenizer, max_len,weights=None):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.rule_ids = rule_ids
        self.weights = weights if weights is not None else [1.0]*len(texts)

    def __len__(self): return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        enc = self.tokenizer(
            text, padding='max_length', truncation=True, max_length=self.max_len, return_tensors="pt"
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.float)
        item['rule_ids']= torch.tensor(self.rule_ids[idx])
        item['weights'] = torch.tensor(self.weights[idx], dtype=torch.float)
        return item

In [ ]:
class JigsawModel(nn.Module):
    def __init__(self, model_path):
        super().__init__()
        self.base = AutoModel.from_pretrained(model_path)
        self.drop = nn.Dropout(0.15)
        self.out = nn.Linear(self.base.config.hidden_size, 1)

    def forward(self, input_ids, attention_mask):
        outputs = self.base(input_ids=input_ids, attention_mask=attention_mask)
        pooled = outputs.last_hidden_state[:, 0]
        return self.out(self.drop(pooled)).squeeze(1)

In [ ]:
def train_one_epoch(model, loader, optimizer, scheduler, device):
    model.train()
    total_loss = 0
    for batch in tqdm(loader, desc='Training'):
        optimizer.zero_grad()
        input_ids = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        logits = model(input_ids, mask)
        weights = batch["weights"].to(device)
        loss = nn.BCEWithLogitsLoss(reduction='none')(logits, labels)
        loss = (loss * weights).mean()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
        optimizer.step()
        if scheduler:
            scheduler.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def validate(model, loader, device):
    model.eval()
    preds, targets, rule_ids_list = [], [], []
    total_loss = 0
    criterion = nn.BCEWithLogitsLoss()
    
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            rule_ids = batch["rule_ids"]
            
            logits = model(input_ids, mask)
            loss = criterion(logits, labels)
            total_loss += loss.item()
            
            preds.extend(torch.sigmoid(logits).cpu().numpy())
            targets.extend(labels.cpu().numpy())
            rule_ids_list.extend(rule_ids.cpu().numpy() if torch.is_tensor(rule_ids) else rule_ids)
    
    preds = np.array(preds)
    targets = np.array(targets)
    rule_ids_array = np.array(rule_ids_list)
    
    unique_rules = np.unique(rule_ids_array)
    rule_aucs = {}
    
    for rule_id in unique_rules:
        rule_mask = rule_ids_array == rule_id
        rule_preds = preds[rule_mask]
        rule_targets = targets[rule_mask]
        
        if len(np.unique(rule_targets >= 0.5)) > 1:
            rule_auc = roc_auc_score(rule_targets >= 0.5, rule_preds)
            rule_aucs[rule_id] = rule_auc
        else:
            rule_aucs[rule_id] = np.nan
    
    valid_aucs = [auc for auc in rule_aucs.values() if not np.isnan(auc)]
    avg_auc_per_rule = np.mean(valid_aucs) if valid_aucs else 0
    
    val_loss = total_loss / len(loader)
    return avg_auc_per_rule, val_loss, preds

In [ ]:
def train_model_seed_pseudo(seed, gpu_id):
    import random
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    
    device = torch.device(f"cuda:{gpu_id}")
    print(f"[Seed {seed}] Training on {device}")
    
    train_data = pd.read_csv(f'fixed_train_split_seed_{seed}.csv')
    val_data = pd.read_csv(f'fixed_val_split_seed_{seed}.csv')
    unlabelled_taken = pd.read_csv(f'fixed_pseudo_seed_{seed}.csv')
    
    train_ds = JigsawDataset(
        unlabelled_taken['text'].tolist(), 
        unlabelled_taken['rule_violation'].tolist(), 
        unlabelled_taken['rule_id'].tolist(), 
        tokenizer, MAX_LEN,

    )
    
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    
    model = JigsawModel(MODEL_PATH).to(device)
    for name, param in model.named_parameters():
        if name.startswith('base.embedding'):
            param.requires_grad = False
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5, eps=1e-6,weight_decay=1e-2)
    total_steps = U_EPOCHS * len(train_loader)
    warmup_steps = int(0.1 * total_steps)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps,
    )
    
    for epoch in range(U_EPOCHS):
        print(f"[Seed {seed}] Epoch {epoch+1}/{U_EPOCHS}")
        loss = train_one_epoch(model, train_loader, optimizer, scheduler, device)
        torch.save(model.state_dict(), f"model_seed_{seed}_pseudo.bin")

    return seed, None

In [ ]:
def train_model_seed(seed, gpu_id):
    import random
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    
    device = torch.device(f"cuda:{gpu_id}")
    print(f"[Seed {seed}] Training on {device}")
    
    train_data = pd.read_csv(f'fixed_train_split_seed_{seed}.csv')
    val_data = pd.read_csv(f'fixed_val_split_seed_{seed}.csv')
    unlabelled_taken = pd.read_csv(f'fixed_pseudo_seed_{seed}.csv')
    
    train_ds = JigsawDataset(
        train_data['text'].tolist(),#+unlabelled_taken['text'].tolist(), 
        train_data['label'].tolist(),#+unlabelled_taken['rule_violation'].tolist(), 
        train_data['rule_id'].tolist(),#+unlabelled_taken['rule_id'].tolist(), 
        tokenizer, MAX_LEN,
        [1.0]*len(train_data),# + [.5]*len(unlabelled_taken)

    )
    
    val_ds = JigsawDataset(
        val_data['text'].tolist(), 
        val_data['label'].tolist(), 
        val_data['rule_id'].tolist(), 
        tokenizer, MAX_LEN
    )
    
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)
    
    model = JigsawModel(MODEL_PATH).to(device)
    model.load_state_dict(torch.load(f"model_seed_{seed}_pseudo.bin",device))
    for name, param in model.named_parameters():
        if name.startswith('base.embedding'):
            param.requires_grad = False
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-5, eps=1e-6,weight_decay= 3e-2)
    total_steps = EPOCHS * len(train_loader)
    warmup_steps = int(0.1 * total_steps)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps,
    )
    
    best_auc = 0
    best_loss= None
    for epoch in range(EPOCHS):
        print(f"[Seed {seed}] Epoch {epoch+1}/{EPOCHS}")
        loss = train_one_epoch(model, train_loader, optimizer, scheduler, device)
        val_auc, val_loss, val_preds = validate(model, val_loader, device)
        
        print(f"[Seed {seed}] Loss: {loss:.4f}, Val Loss: {val_loss:.4f}, Val AUC: {val_auc:.4f}")
        if val_auc > best_auc:
            best_auc = val_auc
            best_loss= val_loss
            torch.save(model.state_dict(), f"model_seed_{seed}.bin")
    
    print(f"[Seed {seed}] Best validation AUC: {best_auc:.4f}")
    import json
    with open(f'results_seed_{seed}.json', 'w') as f:
        json.dump({'seed': seed, 'best_auc': best_auc, 'best_loss': best_loss}, f)

    return seed, best_auc

In [ ]:
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    import torch.multiprocessing as mp
    mp.set_start_method('fork', force=True)
    
    processes = []
    for idx, seed in enumerate(SEEDS):
       gpu_id = idx % torch.cuda.device_count()
       p = mp.Process(target=train_model_seed_pseudo, args=(seed, gpu_id))
       p.start()
       processes.append(p)
    
    for p in processes:
       p.join()

    processes = []
    for idx, seed in enumerate(SEEDS):
       gpu_id = idx % torch.cuda.device_count()
       p = mp.Process(target=train_model_seed, args=(seed, gpu_id))
       p.start()
       processes.append(p)
    
    for p in processes:
       p.join()

    import json
    results = []
    for seed in SEEDS:
      with open(f'results_seed_{seed}.json', 'r') as f:
          results.append(json.load(f))
    
    aucs = [r['best_auc'] for r in results]
    losses = [r['best_loss'] for r in results]
    
    print(f"AUC: {np.mean(aucs):.4f} ± {np.std(aucs):.4f}")
    print(f"Loss: {np.mean(losses):.4f} ± {np.std(losses):.4f}")

    print("All models trained!")

In [ ]:
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    df_test = pd.read_csv(test_path)
    df_test["text"] = df_test["rule"] + " [SEP] " + df_test["body"]
    
    test_ds = JigsawDataset(df_test['text'].tolist(), [0]*len(df_test), [0]*len(df_test), tokenizer, MAX_LEN)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE)
    
    all_preds = []
    
    for seed in SEEDS:
        device = torch.device("cuda:0")
        model = JigsawModel(MODEL_PATH).to(device)
        model.load_state_dict(torch.load(f"model_seed_{seed}.bin", map_location=device))
        model.eval()
        
        test_preds = []
        with torch.no_grad():
            for batch in tqdm(test_loader, desc=f"Inference seed {seed}"):
                ids = batch['input_ids'].to(device)
                mask = batch['attention_mask'].to(device)
                logits = model(ids, mask)
                test_preds.extend(torch.sigmoid(logits).cpu().numpy())
        
        all_preds.append(test_preds)
    
    ensemble_preds = np.mean(all_preds, axis=0)
    
    sample = pd.read_csv(sample_sub_path)
    sample["rule_violation"] = ensemble_preds
    sample.to_csv("submission.csv", index=False)
    print(f"Ensembled {len(SEEDS)} models")
else:
    !touch submission.csv
    
!head -n 4 submission.csv